# 03 Model 1: Word Embeddings + Classifier
Training of architecture 1: frozen pretrained word embeddings with a 2-layer classifier (ReLU non-linearity). Only the classifier is trained. Experiments are tracked with Weights & Biases.

Load the features and labels that were saved in `02_preprocessing.ipynb`. The test set is not loaded here: it is only used once at the end, in `05_evaluation.ipynb`.

In [2]:
import numpy as np
import torch

data = np.load("features/model1.npz")
X_train = torch.tensor(data["X_train"])
y_train = torch.tensor(data["y_train"])
X_valid = torch.tensor(data["X_valid"])
y_valid = torch.tensor(data["y_valid"])
print(X_train.shape, X_train.dtype)
print(y_train.shape, y_train.dtype)

torch.Size([16801, 1200]) torch.float32
torch.Size([16801]) torch.int64


## Classifier
The 2-layer classifier like in the course exercise "PyTorch" (Week 1): linear layer, ReLU, linear layer. Input: 1200 features, output: 2 classes (h1 or h2).

In [3]:
import torch.nn as nn

class Classifier(nn.Module):

    def __init__(self, input_dim, output_dim, inner_dim=10):
        super().__init__()
        self.linear1 = nn.Linear(input_dim, inner_dim)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(inner_dim, output_dim)

    def forward(self, x):
        out = self.linear1(x)
        out = self.relu(out)
        out = self.linear2(out)
        return out

torch.manual_seed(42)
model = Classifier(1200, 2, inner_dim=100)
print(model)

Classifier(
  (linear1): Linear(in_features=1200, out_features=100, bias=True)
  (relu): ReLU()
  (linear2): Linear(in_features=100, out_features=2, bias=True)
)


Number of parameters per layer (weights and biases), as in the course exercise "PyTorch" (Week 1).

In [14]:
for name, params in model.named_parameters():
    print(name, params.shape)

linear1.weight torch.Size([100, 1200])
linear1.bias torch.Size([100])
linear2.weight torch.Size([2, 100])
linear2.bias torch.Size([2])


Test the untrained classifier on the first 5 training examples: one pair of scores (h1, h2) per example.

In [15]:
x = X_train[:5]
output = model(x)
print(x.shape)
print(output.shape)
print(output)

torch.Size([5, 1200])
torch.Size([5, 2])
tensor([[-0.1848, -0.1143],
        [-0.1771, -0.0636],
        [-0.1528, -0.0544],
        [-0.1209, -0.0621],
        [-0.1661, -0.0300]], grad_fn=<AddmmBackward0>)


## Loss
Cross-entropy loss for the 2 classes (course slides "Preliminaries", slide 27). `nn.CrossEntropyLoss` applies the softmax to the raw scores itself, so the classifier has no softmax layer. A binary decision can be made with 1 output and a sigmoid or with 2 outputs and a softmax (course slides "Project Discussions" (Week 5), slide 22). 2 outputs are used, one score per hypothesis, as in the classifier of the course exercise "PyTorch", so the prediction is the hypothesis with the higher score (`argmax`). A softmax layer in the model would apply the softmax twice (the "double sigmoid" warning on slide 30 of the same slides, Week 6). Before training, the loss should be about -log(0.5) ≈ 0.69, the value of random guessing between 2 classes.

In [16]:
criterion = nn.CrossEntropyLoss()

y = y_train[:5]
loss = criterion(output, y)
print(y)
print(loss)

tensor([1, 1, 0, 0, 1])
tensor(0.6781, grad_fn=<NllLossBackward0>)


## One training step 
The Adam optimizer updates the weights (course slides "Preliminaries", slide 29). Adam is used rather than AdamW (course slides "Project Discussions" (Week 6), slide 31): AdamW only changes how weight decay is applied, and no weight decay is used here, so both would make the same update steps (algorithm in the PyTorch documentation `torch.optim.AdamW`; Loshchilov & Hutter, 2019). One training step on the 5 examples as in the course exercise "PyTorch" (Week 1): forward pass, loss, backpropagation, update. Afterwards, the loss on the same 5 examples should be lower.

In [17]:
from torch.optim import Adam

optimizer = Adam(model.parameters(), lr=0.001)

output = model(x)
loss = criterion(output, y)
optimizer.zero_grad()
loss.backward()
optimizer.step()

print(loss)
print(criterion(model(x), y))

tensor(0.6781, grad_fn=<NllLossBackward0>)
tensor(0.6114, grad_fn=<NllLossBackward0>)


## Batches
The data is split into batches with a `DataLoader`, as in the course exercise "Experiment Tracking": training data in batches of 32, shuffled in every epoch. The course recommends the largest batch size that fits into the GPU memory (course slides "Project Discussions" (Week 4), slide 14); here the training runs on the CPU, where memory is not the limit, and one epoch takes only a few seconds, so the batch size of the course exercise is kept. 16801 = 525 x 32 + 1, so the last batch would contain a single example; `drop_last=True` skips it (a different example in every epoch, because of the shuffling). For measuring loss and accuracy, all batches have the same size, so that the average of the batch losses equals the average over all examples: validation data in 10 batches of 100, training data in 53 batches of 317 (16801 = 53 x 317), both not shuffled. `TensorDataset` pairs each row of features with its label.

In [18]:
from torch.utils.data import TensorDataset, DataLoader

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True, drop_last=True)
train_eval_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=317)
valid_loader = DataLoader(TensorDataset(X_valid, y_valid), batch_size=100)

x_batch, y_batch = next(iter(train_loader))
print(len(train_loader), len(train_eval_loader), len(valid_loader))
print(x_batch.shape, y_batch.shape)

525 53 10
torch.Size([32, 1200]) torch.Size([32])


## Evaluation
Loss and accuracy on a dataset, batch by batch, as in the validation of the course exercise "Experiment Tracking": the loss of each batch and the predictions (`argmax`, the class with the higher score) and labels are collected; at the end, the loss is the average of the batch losses and the accuracy is the share of correct predictions over all examples. `model.eval()` and `torch.no_grad()` as in the `test_loop` of the PyTorch tutorial "Optimizing Model Parameters". A new untrained model is tested first: it should be at about 0.69 loss and 50% accuracy.

In [19]:
def evaluate(model, loader):
    model.eval()
    losses, predictions, labels = [], [], []
    with torch.no_grad():
        for x, y in loader:
            output = model(x)
            losses.append(criterion(output, y).item())
            predictions.append(output.argmax(dim=1))
            labels.append(y)
    predictions = torch.cat(predictions)
    labels = torch.cat(labels)
    accuracy = (predictions == labels).float().mean().item()
    return np.mean(losses), accuracy

torch.manual_seed(42)
model = Classifier(1200, 2, inner_dim=100)
valid_loss, valid_acc = evaluate(model, valid_loader)
print(f"Validation loss: {valid_loss:.4f}, accuracy: {valid_acc:.2%}")

Validation loss: 0.6943, accuracy: 49.10%


## Experiment tracking
The training runs are tracked with Weights and Biases (W&B), as in the course exercise "Experiment Tracking". Log in to W&B (the API key was saved once with `wandb login` in the terminal).

In [20]:
import wandb
wandb.login()

True

## Training
Training loop as in the PyTorch tutorial "Optimizing Model Parameters": in every epoch, one training step (as in the course exercise "PyTorch") for each batch of the training data, then evaluation on the training and validation data. The training is written as a function `train()` with the hyperparameters as arguments, as in the course exercise "Experiment Tracking", so that it can be run with different hyperparameters. Every call trains a fresh model with seed 42 and creates one W&B run. Each run is logged to W&B: the hyperparameters in `config` and, after every epoch, the training and validation loss and accuracy with `wandb.log` (with `epoch` for the x-axis). `wandb.finish()` ends the run. After every epoch, the model is saved to `checkpoints/` if its validation loss is the lowest so far (`torch.save` of the `state_dict`, as in the PyTorch tutorial "Save and Load the Model"). At the end, the weights of the best epoch are loaded again, so `train()` returns the best model, not the one after the last epoch. The validation loss is used because it changes more smoothly than the accuracy. In addition, the best validation loss, the validation accuracy of that epoch and the best epoch are written to the run summary in W&B, so that the runs can be sorted by their best validation loss (by default, the summary shows the values of the last epoch). The cell ends with a first run with the default hyperparameters (10 epochs) as a walk-through; the hyperparameter experiments follow in the section "Experiments".

Further decisions: the learning rate stays constant (no learning rate schedule), the simplest choice, which can also work well (course slides "Project Discussions" (Week 6), slide 33, with Hägele et al., 2024). No early stopping: one epoch of model 1 takes only a few seconds, so all epochs are trained and the epoch with the lowest validation loss is kept afterwards; this gives the same model as early stopping with a long enough patience (same slides, slide 38), and the full curves in W&B show the overfitting. Model 2, where one epoch takes 1 to 2 minutes, uses early stopping. No dropout or weight decay (same slides, slide 34): selecting the best epoch by the validation loss already keeps the model from before it overfits (see the conclusion); dropout is tested for model 2 (`04_model2_rnn.ipynb`).

In [21]:
import os

os.makedirs("checkpoints", exist_ok=True)

def train(inner_dim=100, learning_rate=0.001, num_epochs=10):
    name = f"glove-mlp-h{inner_dim}-lr{learning_rate}-e{num_epochs}"
    checkpoint_path = f"checkpoints/{name}.pth"
    wandb.init(
        project="nlp-project1-art",
        name=name,
        config={
            "model": "glove-mlp",
            "inner_dim": inner_dim,
            "learning_rate": learning_rate,
            "epochs": num_epochs,
            "batch_size": train_loader.batch_size,
            "seed": 42,
        },
    )

    torch.manual_seed(42)
    model = Classifier(1200, 2, inner_dim=inner_dim)
    optimizer = Adam(model.parameters(), lr=learning_rate)
    best_valid_loss = float("inf")

    for epoch in range(num_epochs):
        model.train()
        for x, y in train_loader:
            output = model(x)
            loss = criterion(output, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        train_loss, train_acc = evaluate(model, train_eval_loader)
        valid_loss, valid_acc = evaluate(model, valid_loader)
        print(f"Epoch {epoch + 1}: train loss {train_loss:.4f}, train acc {train_acc:.2%}, valid loss {valid_loss:.4f}, valid acc {valid_acc:.2%}")
        wandb.log({"epoch": epoch + 1, "train_loss": train_loss, "train_acc": train_acc, "val_loss": valid_loss, "val_acc": valid_acc})
        if valid_loss < best_valid_loss:
            best_valid_loss = valid_loss
            best_valid_acc = valid_acc
            best_epoch = epoch + 1
            torch.save(model.state_dict(), checkpoint_path)

    wandb.run.summary["best_val_loss"] = best_valid_loss
    wandb.run.summary["best_val_acc"] = best_valid_acc
    wandb.run.summary["best_epoch"] = best_epoch
    wandb.finish()
    print(f"Best epoch: {best_epoch}, valid loss {best_valid_loss:.4f}, valid acc {best_valid_acc:.2%}")
    model.load_state_dict(torch.load(checkpoint_path, weights_only=True))
    return model

model = train(inner_dim=100, learning_rate=0.001)

Epoch 1: train loss 0.6693, train acc 60.63%, valid loss 0.6751, valid acc 58.70%
Epoch 2: train loss 0.6449, train acc 62.09%, valid loss 0.6654, valid acc 57.30%
Epoch 3: train loss 0.6251, train acc 64.97%, valid loss 0.6576, valid acc 60.50%
Epoch 4: train loss 0.6187, train acc 65.10%, valid loss 0.6590, valid acc 59.60%
Epoch 5: train loss 0.6051, train acc 66.71%, valid loss 0.6511, valid acc 61.50%
Epoch 6: train loss 0.6058, train acc 66.32%, valid loss 0.6654, valid acc 59.90%
Epoch 7: train loss 0.5844, train acc 68.74%, valid loss 0.6530, valid acc 62.20%
Epoch 8: train loss 0.5737, train acc 69.86%, valid loss 0.6494, valid acc 62.70%
Epoch 9: train loss 0.5630, train acc 70.32%, valid loss 0.6572, valid acc 60.20%
Epoch 10: train loss 0.5458, train acc 72.52%, valid loss 0.6500, valid acc 63.30%


epoch,▁▂▃▃▄▅▆▆▇█
train_acc,▁▂▄▄▅▄▆▆▇█
train_loss,█▇▅▅▄▄▃▃▂▁
val_acc,▃▁▅▄▆▄▇▇▄█
val_loss,█▅▃▄▁▅▂▁▃▁
best_epoch,8
best_val_acc,0.627
best_val_loss,0.64944
epoch,10
train_acc,0.72519
train_loss,0.54578


Best epoch: 8, valid loss 0.6494, valid acc 62.70%


Check that the returned model is the one of the best epoch: its validation loss and accuracy must be the same as in that epoch.

In [22]:
valid_loss, valid_acc = evaluate(model, valid_loader)
print(f"Validation loss: {valid_loss:.4f}, accuracy: {valid_acc:.2%}")

Validation loss: 0.6494, accuracy: 62.70%


## Experiments
Hyperparameter tuning on the validation set (course slides "Course Project HS26", slide 10: the validation set is used to compare model settings and to select the best model and hyperparameters). The experiments answer two questions: how large does the classifier have to be (hidden size), and which learning rate works, "the most important hyperparameter" (course slides "Project Discussions" (Week 6), slide 32)? The hidden size of the classifier (50, 100, 300) and the learning rate (0.001, 0.01) are varied, each for 20 epochs. In every run, the best epoch is selected by the validation loss. Because the seed is set inside `train()`, the two runs with the same hidden size start from the same initial weights and see the batches in the same order, so the difference between them comes only from the learning rate. Each configuration is trained once. With seed 42, a run is exactly reproducible (the first 10 epochs of the 10-epoch run above and of the 20-epoch run with the same settings are identical), but the differences between configurations can partly be chance; the conclusion compares them with the standard error of the validation accuracy.

In [23]:
for inner_dim in [50, 100, 300]:
    for learning_rate in [0.001, 0.01]:
        train(inner_dim=inner_dim, learning_rate=learning_rate, num_epochs=20)

Epoch 1: train loss 0.6716, train acc 59.84%, valid loss 0.6774, valid acc 58.10%
Epoch 2: train loss 0.6465, train acc 63.16%, valid loss 0.6638, valid acc 58.20%
Epoch 3: train loss 0.6361, train acc 63.41%, valid loss 0.6631, valid acc 59.70%
Epoch 4: train loss 0.6242, train acc 65.13%, valid loss 0.6533, valid acc 61.00%
Epoch 5: train loss 0.6180, train acc 65.81%, valid loss 0.6509, valid acc 60.40%
Epoch 6: train loss 0.6158, train acc 65.93%, valid loss 0.6554, valid acc 60.10%
Epoch 7: train loss 0.6127, train acc 66.07%, valid loss 0.6527, valid acc 61.00%
Epoch 8: train loss 0.6175, train acc 65.13%, valid loss 0.6656, valid acc 60.40%
Epoch 9: train loss 0.6164, train acc 65.08%, valid loss 0.6538, valid acc 61.90%
Epoch 10: train loss 0.6096, train acc 66.24%, valid loss 0.6616, valid acc 60.10%
Epoch 11: train loss 0.6034, train acc 66.74%, valid loss 0.6574, valid acc 60.20%
Epoch 12: train loss 0.6001, train acc 66.73%, valid loss 0.6554, valid acc 62.20%
Epoch 13: tra

epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
train_acc,▁▃▄▅▅▅▅▅▅▆▆▆▇▆▇▇▆▇█▇
train_loss,█▆▆▅▄▄▄▄▄▄▃▃▃▃▂▂▂▂▁▁
val_acc,▁▁▄▆▅▄▆▅▇▄▅█▇▆▇▅▅▅▇▅
val_loss,█▄▄▂▁▂▂▅▂▄▃▂▃▄▁▃▄▇▂▄
best_epoch,15
best_val_acc,0.619
best_val_loss,0.65046
epoch,20
train_acc,0.68639
train_loss,0.57781


Best epoch: 15, valid loss 0.6505, valid acc 61.90%


Epoch 1: train loss 0.6941, train acc 50.36%, valid loss 0.6937, valid acc 50.80%
Epoch 2: train loss 0.6932, train acc 49.64%, valid loss 0.6933, valid acc 49.20%
Epoch 3: train loss 0.6931, train acc 50.36%, valid loss 0.6930, valid acc 50.80%
Epoch 4: train loss 0.6934, train acc 50.36%, valid loss 0.6931, valid acc 50.80%
Epoch 5: train loss 0.6937, train acc 50.36%, valid loss 0.6934, valid acc 50.80%
Epoch 6: train loss 0.6934, train acc 50.36%, valid loss 0.6932, valid acc 50.80%
Epoch 7: train loss 0.6931, train acc 49.66%, valid loss 0.6933, valid acc 49.20%
Epoch 8: train loss 0.6931, train acc 49.66%, valid loss 0.6932, valid acc 49.20%
Epoch 9: train loss 0.6931, train acc 49.66%, valid loss 0.6933, valid acc 49.20%
Epoch 10: train loss 0.6935, train acc 49.66%, valid loss 0.6939, valid acc 49.20%
Epoch 11: train loss 0.6931, train acc 49.66%, valid loss 0.6932, valid acc 49.20%
Epoch 12: train loss 0.6933, train acc 49.66%, valid loss 0.6935, valid acc 49.20%
Epoch 13: tra

epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
train_acc,█▁████▁▁▁▁▁▁▁█▁███▁█
train_loss,█▂▂▃▅▃▂▁▂▄▁▃▃▁▆▁▂▂▄▂
val_acc,█▁████▁▁▁▁▁▁▁█▁███▁█
val_loss,▅▃▁▂▃▂▃▂▃▆▂▄▄▁█▁▁▁▆▁
best_epoch,3
best_val_acc,0.508
best_val_loss,0.69302
epoch,20
train_acc,0.5036
train_loss,0.69311


Best epoch: 3, valid loss 0.6930, valid acc 50.80%


... (8 outputs left out)

Best epoch: 3, valid loss 0.6550, valid acc 60.20%


Epoch 1: train loss 0.6927, train acc 50.40%, valid loss 0.6928, valid acc 50.80%
Epoch 2: train loss 0.6733, train acc 56.44%, valid loss 0.6905, valid acc 55.10%
Epoch 3: train loss 0.6372, train acc 63.92%, valid loss 0.6645, valid acc 58.70%
Epoch 4: train loss 0.6226, train acc 65.50%, valid loss 0.6573, valid acc 61.10%
Epoch 5: train loss 0.6232, train acc 65.69%, valid loss 0.6575, valid acc 59.20%
Epoch 6: train loss 0.6277, train acc 64.40%, valid loss 0.6662, valid acc 60.20%
Epoch 7: train loss 0.6104, train acc 66.74%, valid loss 0.6540, valid acc 62.70%
Epoch 8: train loss 0.6143, train acc 66.05%, valid loss 0.6611, valid acc 62.00%
Epoch 9: train loss 0.6384, train acc 63.33%, valid loss 0.6907, valid acc 58.20%
Epoch 10: train loss 0.6051, train acc 67.13%, valid loss 0.6563, valid acc 62.40%
Epoch 11: train loss 0.6190, train acc 65.35%, valid loss 0.6716, valid acc 57.90%
Epoch 12: train loss 0.6013, train acc 67.56%, valid loss 0.6592, valid acc 62.00%
Epoch 13: tra

epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
train_acc,▁▃▆▇▇▆▇▇▆█▇█▇█▇█████
train_loss,█▇▄▃▃▄▂▃▄▂▃▂▂▂▂▁▁▂▁▂
val_acc,▁▃▆▇▆▆█▇▅█▅▇▆█▇█▇██▇
val_loss,██▃▂▂▃▁▂█▁▄▂▂▄▆▄▄▄▇▂
best_epoch,7
best_val_acc,0.627
best_val_loss,0.65395
epoch,20
train_acc,0.68157
train_loss,0.59756


Best epoch: 7, valid loss 0.6540, valid acc 62.70%


## Results
Load the best checkpoint of every experiment, evaluate it on the validation set, and sort the results by the validation loss (lowest first).

In [25]:
results = []
for inner_dim in [50, 100, 300]:
    for learning_rate in [0.001, 0.01]:
        model = Classifier(1200, 2, inner_dim=inner_dim)
        model.load_state_dict(torch.load(f"checkpoints/glove-mlp-h{inner_dim}-lr{learning_rate}-e20.pth", weights_only=True))
        valid_loss, valid_acc = evaluate(model, valid_loader)
        results.append((valid_loss, valid_acc, inner_dim, learning_rate))

for valid_loss, valid_acc, inner_dim, learning_rate in sorted(results):
    print(f"h{inner_dim}, lr {learning_rate}: valid loss {valid_loss:.4f}, valid acc {valid_acc:.2%}")

h100, lr 0.001: valid loss 0.6494, valid acc 62.70%
h50, lr 0.001: valid loss 0.6505, valid acc 61.90%
h100, lr 0.01: valid loss 0.6524, valid acc 62.00%
h300, lr 0.01: valid loss 0.6540, valid acc 62.70%
h300, lr 0.001: valid loss 0.6550, valid acc 60.20%
h50, lr 0.01: valid loss 0.6930, valid acc 50.80%


## Dead ReLU units
With learning rate 0.01 and hidden size 50, the model predicts the same class for every example. Count how many hidden units are active, i.e. output a value greater than 0 after the ReLU for more than 1% of the training examples (a unit that fires on only one or two of the 16801 examples is counted as dead): before training, and in the best checkpoints of all six experiments. The untrained models are created with seed 42, so they have the same start weights as the training runs with the same hidden size. A unit that outputs 0 for every input does not learn any more, because its gradient is 0 ("dying ReLU", Lu et al., 2020).

In [4]:
# no training here, so no gradients are needed
with torch.no_grad():
    for inner_dim in [50, 100, 300]:
        # same start weights as the training runs with this hidden size
        torch.manual_seed(42)
        untrained = Classifier(1200, 2, inner_dim=inner_dim)
        # hidden layer only (linear1 + ReLU): one row per example, one column per unit
        hidden = untrained.relu(untrained.linear1(X_train))
        # share of the examples on which each unit outputs more than 0; active if more than 1%
        active = (hidden > 0).float().mean(dim=0) > 0.01
        # count the active units (True counts as 1)
        print(f"h{inner_dim}, untrained: {active.sum().item()} of {inner_dim} units active")
        # the same for the trained models
        for learning_rate in [0.001, 0.01]:
            # load the weights of the best epoch of the run (-e20 = the run had 20 epochs)
            model = Classifier(1200, 2, inner_dim=inner_dim)
            model.load_state_dict(torch.load(f"checkpoints/glove-mlp-h{inner_dim}-lr{learning_rate}-e20.pth", weights_only=True))
            hidden = model.relu(model.linear1(X_train))
            active = (hidden > 0).float().mean(dim=0) > 0.01
            print(f"h{inner_dim}, lr {learning_rate}: {active.sum().item()} of {inner_dim} units active")

h50, untrained: 49 of 50 units active
h50, lr 0.001: 5 of 50 units active
h50, lr 0.01: 0 of 50 units active
h100, untrained: 94 of 100 units active
h100, lr 0.001: 17 of 100 units active
h100, lr 0.01: 1 of 100 units active
h300, untrained: 279 of 300 units active
h300, lr 0.001: 40 of 300 units active
h300, lr 0.01: 1 of 300 units active


## Conclusion
- **Chosen configuration:** hidden size 100, learning rate 0.001 (best epoch 8): validation loss 0.6494, accuracy 62.70%. Its checkpoint `checkpoints/glove-mlp-h100-lr0.001-e20.pth` is evaluated once on the test set in `05_evaluation.ipynb`.
- **The hyperparameters barely matter:** all configurations that learned are between 0.6494 and 0.6550 validation loss and between 60.20% and 62.70% accuracy. With 1000 validation examples, the standard error of an accuracy of about 62% is √(0.62 × 0.38 / 1000) ≈ 1.5 percentage points, so these differences are within noise. The limit is the input, not the classifier: the average word vectors hide the few words in which h1 and h2 differ (see `01_data_exploration.ipynb`).
- **Learning rate 0.01 is too high:** with hidden size 50, no hidden unit is active any more (0 of 50), so the model predicts h1 for every example (50.80%, the share of h1 in the validation set); with hidden sizes 100 and 300, only 1 unit remains active, and with hidden size 100 the loss stayed at 0.693 (guessing) for 8 epochs before it started to learn.
- **Few units are enough:** with learning rate 0.001, only 5 of 50, 17 of 100 and 40 of 300 units remain active. A model with a single active unit (hidden size 100 or 300, learning rate 0.01) still reaches 62.00% and 62.70%, as much as the best model. The averaged input apparently provides very few useful features, which again points to the input as the limit.
- **Larger hidden sizes overfit faster:** with hidden size 300 and learning rate 0.001, the training accuracy reaches 94.71% after 20 epochs, while the validation loss rises from 0.6550 (best epoch 3) to 0.9564. Selecting the best epoch by the validation loss keeps the model from before the overfitting.

## Sources
Course materials: NLP HS26 by A. Marfurt, Lucerne University of Applied Sciences and Arts (HSLU), 2026. Full references in README.md.
- Loading the features: `np.load` as in the NumPy documentation, https://numpy.org/doc/stable/reference/generated/numpy.savez.html.
- Tensors: `torch.tensor` as in the course exercise "PyTorch" (Week 1).
- Naming: `X` (features, uppercase because it is a matrix) and `y` (labels) following the scikit-learn convention, https://scikit-learn.org/stable/glossary.html.
- Classifier: class `MyNetwork` from the course exercise "PyTorch" (Week 1), renamed to `Classifier`, based on the PyTorch tutorial "Neural Networks", https://docs.pytorch.org/tutorials/beginner/blitz/neural_networks_tutorial.html; architecture "linear layer → ReLU → linear layer" from the course slides "Preliminaries" (Week 1), slide 25.
- Parameters and test forward pass: `named_parameters()` and a forward pass with a small batch as in the course exercise "PyTorch" (Week 1).
- Loss: cross-entropy from the course slides "Preliminaries" (Week 1), slide 27; `criterion` and `loss = criterion(output, target)` as in the course exercise "PyTorch" (Week 1); `nn.CrossEntropyLoss` (applies the softmax to the raw scores) from the PyTorch documentation, https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html.
- Random seed: fixed seed 42 as in the course exercise "Experiment Tracking" (`seed_everything(42)`); `torch.manual_seed` from the PyTorch documentation "Reproducibility", https://docs.pytorch.org/docs/stable/notes/randomness.html.
- Optimizer and training step: Adam from the course slides "Preliminaries" (Week 1), slide 29; `optimizer.zero_grad()`, `loss.backward()`, `optimizer.step()` as in the course exercise "PyTorch" (Week 1).
- Batches: `DataLoader` with `batch_size=32` and `shuffle=True` for training and no shuffling for validation as in the course exercise "Experiment Tracking"; `TensorDataset` and `drop_last=True` from the PyTorch documentation, https://docs.pytorch.org/docs/stable/data.html.
- Evaluation: collecting the batch losses, `argmax` predictions and labels, joining them with `torch.cat` and averaging the batch losses as in the validation of the course exercise "Experiment Tracking" (`validation_step` and `on_validation_epoch_end`); `model.eval()` and `torch.no_grad()` as in `test_loop` of the PyTorch tutorial "Optimizing Model Parameters", https://docs.pytorch.org/tutorials/beginner/basics/optimization_tutorial.html; `np.mean` from NumPy (Harris et al., 2020).
- Training loop: epochs with a train loop over all batches and an evaluation after each epoch, and `model.train()`, as in the PyTorch tutorial "Optimizing Model Parameters"; the training step inside as in the course exercise "PyTorch" (Week 1); training as a function `train()` with the hyperparameters as arguments (with default values) and the seed set inside, as in the course exercise "Experiment Tracking".
- Experiment tracking: `wandb.login()`, `wandb.init(project=..., config=...)` and `wandb.finish()` as in the course exercise "Experiment Tracking"; run name `name=` from the W&B `wandb.init()` reference, https://docs.wandb.ai/models/ref/python/functions/init; logging the metrics once per epoch with `epoch` as x-axis as in the W&B documentation "Log objects and media", https://docs.wandb.ai/models/track/log (written there as `run.log`, which is the same as `wandb.log` for the current run); `train_loader.batch_size` from the PyTorch documentation, https://docs.pytorch.org/docs/stable/data.html; Weights and Biases (Biewald, 2020); `nbformat` installed for W&B, as in the course exercise "Experiment Tracking".
- Saving the best model: `torch.save(model.state_dict(), ...)` and `model.load_state_dict(torch.load(..., weights_only=True))` as in the PyTorch tutorial "Save and Load the Model", https://docs.pytorch.org/tutorials/beginner/basics/saveloadrun_tutorial.html; `os.makedirs` from the Python standard library, https://docs.python.org/3/library/os.html; `float("inf")` from the Python documentation, https://docs.python.org/3/library/functions.html#float; best values in the run summary (`run.summary[...]`, here `wandb.run.summary` for the current run) as in the W&B documentation "Log summary metrics", https://docs.wandb.ai/models/track/log/log-summary.
- Experiments: hyperparameter tuning on the validation set, as on slide 10 of the course slides "Course Project HS26" (compare model settings, select the best model and hyperparameters).
- Results: loading the checkpoints with `load_state_dict` as in the PyTorch tutorial "Save and Load the Model"; sorting the results with `sorted()` (tuples are compared by their first item first) as in the Python documentation "Sorting Techniques", https://docs.python.org/3/howto/sorting.html.
- Dead ReLU units: term and explanation "dying ReLU" from Lu, Shin, Su & Karniadakis (2020), *Dying ReLU and Initialization: Theory and Numerical Examples*, Communications in Computational Physics, https://doi.org/10.4208/cicp.OA-2020-0165; `torch.no_grad()` as in the PyTorch tutorial "Optimizing Model Parameters"; share of examples per unit with `.float().mean(dim=0)` as in the evaluation (`torch.mean`, PyTorch documentation, https://docs.pytorch.org/docs/stable/generated/torch.mean.html); counting the active units with `.sum().item()` as in `test_loop` of the PyTorch tutorial "Optimizing Model Parameters".
- Conclusion: standard error of an accuracy √(p(1 − p)/n), from the normal approximation interval in "Binomial proportion confidence interval", Wikipedia, https://en.wikipedia.org/wiki/Binomial_proportion_confidence_interval.
- Training decisions: course slides "Project Discussions": 1 output with a sigmoid or 2 outputs with a softmax (Week 5, slide 22), "double sigmoid" (Week 6, slide 30), Adam and AdamW (Week 6, slide 31), the learning rate as the most important hyperparameter (Week 6, slide 32), a constant learning rate (Week 6, slide 33, citing Hägele, A., Bakouch, E., Kosson, A., Ben Allal, L., Von Werra, L., & Jaggi, M. (2024), *Scaling Laws and Compute-Optimal Training Beyond Fixed Training Durations*, https://arxiv.org/abs/2405.18392), regularization (Week 6, slide 34), early stopping (Week 6, slide 38), batch size (Week 4, slide 14). AdamW: algorithm in the PyTorch documentation `torch.optim.AdamW`, https://docs.pytorch.org/docs/stable/generated/torch.optim.AdamW.html, from Loshchilov & Hutter (2019), *Decoupled Weight Decay Regularization*, ICLR 2019, https://arxiv.org/abs/1711.05101.
- Tools: code written with the help of Claude (Anthropic); all code reviewed and understood by me.